# ✈️ Flight Option Finder — Agentic AI Demo

**Project T10** — CSE476 AI Agent Assignment

This notebook demonstrates a **real AI agent** (not a chatbot) that:
1. Searches for flights using `search_flights(from, to)`
2. Compares prices using `compare_price(options, budget)`
3. Remembers user preferences across turns
4. Makes decisions based on budget, non-stop preference, and price
5. Always provides a second-best fallback

## Setup

In [1]:
import sys, os

# Ensure project root is on path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.agent.flight_agent import FlightAgent
from src.agent.memory import ConversationMemory
from src.tools.search_flights import search_flights
from src.tools.compare_price import compare_price

print("✅ All imports successful")
print(f"   Project root: {project_root}")

✅ All imports successful
   Project root: C:\Users\phani\Downloads\Agentic-ai project


## Trace Visualization Helper

This function renders the agent's multi-step trace as a clear table.

In [2]:
from tabulate import tabulate

def show_trace(state):
    """Display the agent trace as a formatted table."""
    rows = []
    for t in state.trace:
        tool = t.tool or "—"
        inp = str(t.input)[:60] if t.input else "—"
        result = str(t.result)[:70] if t.result else "—"
        decision = str(t.decision)[:50] if t.decision else "—"
        rows.append([t.step, t.action, tool, inp, result, decision])

    print(tabulate(
        rows,
        headers=["Step", "Action", "Tool", "Input", "Result", "Decision"],
        tablefmt="grid",
        maxcolwidths=[5, 18, 16, 40, 50, 35],
    ))

def show_recommendation(agent, state):
    """Display the agent's final recommendation."""
    print("\n" + "=" * 70)
    print(agent.get_response_text(state))
    print("=" * 70)

def show_memory(agent):
    """Show current memory state."""
    print(f"\n📝 Memory: {agent.memory.summary_str()}")
    print(f"   Turn count: {agent.memory.turn_count}")

print("✅ Helper functions loaded")

✅ Helper functions loaded


---
## Scenario 1: Successful Recommendation

**Goal:** Find a non-stop flight from Delhi to Mumbai under ₹6,000.

The agent will:
1. Retrieve memory (empty on first turn)
2. Parse the request to extract: origin=DEL, destination=BOM, budget=6000, non-stop=True
3. Create an execution plan
4. Call `search_flights("DEL", "BOM")` → returns 10 flights
5. Observe results and filter by budget (≤₹6,000)
6. Call `compare_price()` on remaining candidates
7. Apply non-stop preference scoring
8. Select best flight + second-best fallback
9. Update memory

In [3]:
# Create a fresh agent
agent = FlightAgent()

# Run the agent
goal = "Find a flight from Delhi to Mumbai under 6000. I prefer non-stop."
print(f"🎯 User Goal: \"{goal}\"")
print()

state = agent.run(goal)

# Show the full multi-step trace
print("📊 Agent Execution Trace:")
print()
show_trace(state)

# Show recommendation
show_recommendation(agent, state)

# Show memory state after this turn
show_memory(agent)

🎯 User Goal: "Find a flight from Delhi to Mumbai under 6000. I prefer non-stop."

📊 Agent Execution Trace:

+--------+-----------------+----------------+------------------------------------------+--------------------------------------------------+-------------------------------------+
|   Step | Action          | Tool           | Input                                    | Result                                           | Decision                            |
+========+=================+================+==========================================+==================================================+=====================================+
|      1 | retrieve_memory | memory         | session (turn 1)                         | Empty (no prior preferences)                     | Use remembered preferences for this |
|        |                 |                |                                          |                                                  | turn                                |
+-

### Trace Analysis — Scenario 1

The trace above shows the agent performed **8 steps**:
1. **retrieve_memory** → Empty (first turn)
2. **parse_request** → Extracted origin, destination, budget, non-stop
3. **plan** → Created execution plan
4. **tool_call (search_flights)** → Found flights on DEL→BOM route
5. **observe** → Filtered by budget
6. **tool_call (compare_price)** → Compared prices and identified cheapest
7. **decision** → Selected best flight respecting budget + non-stop preference
8. **memory_update** → Stored preferences for future turns

Both tools (`search_flights`, `compare_price`) were actually called and their results directly influenced the decision.

---
## Scenario 2: Memory Across Turns

This scenario demonstrates that memory persists across multiple turns.

**Turn 1:** Set preferences: "I usually fly Delhi to Mumbai. My budget is ₹6,000. Prefer non-stop."

**Turn 2:** Simply say "Find another suitable flight" — the agent must use remembered preferences.

### Turn 1: Set preferences

In [4]:
# Fresh agent for this scenario
agent2 = FlightAgent()

# Turn 1: set preferences and search
turn1_goal = "I want to fly Delhi to Mumbai under 6000. Prefer non-stop."
print(f"🎯 Turn 1: \"{turn1_goal}\"")
print()

state1 = agent2.run(turn1_goal)
show_trace(state1)
show_recommendation(agent2, state1)
show_memory(agent2)

🎯 Turn 1: "I want to fly Delhi to Mumbai under 6000. Prefer non-stop."

+--------+-----------------+----------------+-----------------------------------------+--------------------------------------------------+-------------------------------------+
|   Step | Action          | Tool           | Input                                   | Result                                           | Decision                            |
+========+=================+================+=========================================+==================================================+=====================================+
|      1 | retrieve_memory | memory         | session (turn 1)                        | Empty (no prior preferences)                     | Use remembered preferences for this |
|        |                 |                |                                         |                                                  | turn                                |
+--------+-----------------+---------------

### Turn 2: Memory-driven search

Now the user simply says "Find another suitable flight" without repeating any preferences.
The agent must **read memory** and apply the stored budget (₹6,000), route (DEL→BOM), and non-stop preference.

In [5]:
# Turn 2: rely entirely on memory
turn2_goal = "Find another suitable flight"
print(f"🎯 Turn 2: \"{turn2_goal}\"")
print()

state2 = agent2.run(turn2_goal)
show_trace(state2)

# Verify memory was used
print()
print("🔍 Evidence that memory influenced Turn 2:")
print(f"   Origin used: {state2.origin} (from memory)")
print(f"   Destination used: {state2.destination} (from memory)")
print(f"   Budget used: ₹{state2.budget:,.0f} (from memory)")
print(f"   Non-stop pref: {state2.non_stop} (from memory)")

show_recommendation(agent2, state2)
show_memory(agent2)

🎯 Turn 2: "Find another suitable flight"

+--------+-----------------+----------------+-----------------------------------------+--------------------------------------------------+-------------------------------------+
|   Step | Action          | Tool           | Input                                   | Result                                           | Decision                            |
+========+=================+================+=========================================+==================================================+=====================================+
|      1 | retrieve_memory | memory         | session (turn 2)                        | Budget=₹6,000, Origin=DEL, Destination=BOM, Non- | Use remembered preferences for this |
|        |                 |                |                                         | stop=Yes, Last flight=                           | turn                                |
+--------+-----------------+----------------+----------------------------

### Memory Verification

In the trace for Turn 2:
- Step 1 (**retrieve_memory**) shows the agent read back: Budget=₹6,000, Origin=DEL, Destination=BOM, Non-stop=Yes
- The agent used these remembered values to search and filter — it didn't ask the user to repeat them
- This is **genuine cross-turn memory**, not just printing stored values

---
## Scenario 3: Failure — No Flight Under Budget

**Goal:** Find a non-stop Delhi to Mumbai flight under ₹2,000.

No flight in the dataset costs less than ₹2,000. The agent must:
1. Search for flights
2. Filter by budget → 0 candidates
3. Report honestly that no option fits the budget
4. Show the closest available option and how far over budget it is

In [6]:
# Fresh agent for failure scenario
agent3 = FlightAgent()

goal3 = "Find a non-stop Delhi to Mumbai flight under 2000"
print(f"🎯 User Goal: \"{goal3}\"")
print()

state3 = agent3.run(goal3)

print("📊 Agent Execution Trace:")
print()
show_trace(state3)

# Show the honest failure
show_recommendation(agent3, state3)

# Verify the agent handled it properly
print()
print("🔍 Failure handling verification:")
print(f"   Status: {state3.status}")
print(f"   Selected flight: {state3.selected_flight}")  # Should be None
print(f"   Fallback provided: {state3.fallback_flight is not None}")
if state3.fallback_flight:
    fb = state3.fallback_flight
    print(f"   Fallback: {fb['flight_id']} at ₹{fb['price']:,.0f}")
    print(f"   Over budget by: ₹{fb['price'] - 2000:,.0f}")

🎯 User Goal: "Find a non-stop Delhi to Mumbai flight under 2000"

📊 Agent Execution Trace:

+--------+-----------------+----------------+-----------------------------------------+--------------------------------------------------+-------------------------------------+
|   Step | Action          | Tool           | Input                                   | Result                                           | Decision                            |
+========+=================+================+=========================================+==================================================+=====================================+
|      1 | retrieve_memory | memory         | session (turn 1)                        | Empty (no prior preferences)                     | Use remembered preferences for this |
|        |                 |                |                                         |                                                  | turn                                |
+--------+-------------

### Failure Analysis

The agent:
1. ✅ Searched for flights (tool was actually called)
2. ✅ Compared prices (tool was actually called)
3. ✅ Determined no valid option exists under ₹2,000
4. ✅ Reported honestly instead of hallucinating a result
5. ✅ Provided the closest available option as a fallback
6. ✅ Showed how far over budget the cheapest option is

---
## Tool Verification — Direct Tool Calls

Below we call both tools directly to verify they work independently.

In [7]:
# Tool 1: search_flights
print("═══ Tool 1: search_flights ═══")
print()
flights = search_flights("DEL", "BOM")
print(f"Found {len(flights)} flights on DEL → BOM:")
print()
for f in flights:
    stops = "non-stop" if f["stops"] == 0 else f"{f['stops']} stop(s)"
    print(f"  {f['flight_id']:6s} | {f['airline']:20s} | {f['departure']}-{f['arrival']} | ₹{f['price']:>6,.0f} | {stops}")

═══ Tool 1: search_flights ═══

Found 10 flights on DEL → BOM:

  AI101  | Air India            | 06:00-08:05 | ₹ 7,800 | non-stop
  AI203  | Air India            | 08:00-10:10 | ₹ 7,450 | non-stop
  6E412  | IndiGo               | 14:30-16:45 | ₹ 5,500 | non-stop
  UK911  | Vistara              | 09:15-11:30 | ₹ 8,200 | non-stop
  SG101  | SpiceJet             | 10:00-13:30 | ₹ 4,800 | 1 stop(s)
  AI405  | Air India            | 19:00-21:15 | ₹ 7,600 | non-stop
  6E881  | IndiGo               | 06:15-08:30 | ₹ 5,200 | non-stop
  IX501  | Air India Express    | 07:45-11:00 | ₹ 4,200 | 1 stop(s)
  G8234  | Go First             | 15:50-18:05 | ₹ 6,950 | non-stop
  UK802  | Vistara              | 11:00-13:15 | ₹ 9,500 | non-stop


In [8]:
# Tool 2: compare_price
print("═══ Tool 2: compare_price ═══")
print()
comparison = compare_price(flights, budget=6000)
print(f"Summary: {comparison['summary']}")
print()
print(f"Cheapest:       {comparison['cheapest']['flight_id']} — ₹{comparison['cheapest']['price']:,.0f}")
if comparison['second_best']:
    print(f"Second-best:    {comparison['second_best']['flight_id']} — ₹{comparison['second_best']['price']:,.0f}")
    print(f"Price diff:     ₹{comparison['price_difference']:,.0f}")
print(f"Under budget:   {len(comparison['under_budget'])} flight(s)")
print(f"Nonstop under:  {len(comparison['nonstop_under_budget'])} flight(s)")

═══ Tool 2: compare_price ═══

Summary: Cheapest: IX501 (Air India Express) at ₹4,200. Second-best: SG101 (SpiceJet) at ₹4,800 (₹600 more). Under budget (₹6,000): 4 option(s), of which 2 are non-stop.

Cheapest:       IX501 — ₹4,200
Second-best:    SG101 — ₹4,800
Price diff:     ₹600
Under budget:   4 flight(s)
Nonstop under:  2 flight(s)


---
## Full JSON Trace (Scenario 1)

For grading verification, here is the complete structured trace.

In [9]:
import json

# Re-use state from scenario 1
trace_dicts = state.get_trace_dicts()
print(json.dumps(trace_dicts, indent=2, default=str))

[
  {
    "step": 1,
    "action": "retrieve_memory",
    "tool": "memory",
    "input": "session (turn 1)",
    "result": "Empty (no prior preferences)",
    "decision": "Use remembered preferences for this turn"
  },
  {
    "step": 2,
    "action": "parse_request",
    "tool": "planner",
    "input": "Find a flight from Delhi to Mumbai under 6000. I prefer non-stop.",
    "result": {
      "budget": 6000.0,
      "non_stop": true,
      "origin": "DEL",
      "destination": "BOM"
    },
    "decision": "Extracted user intent from natural language"
  },
  {
    "step": 3,
    "action": "plan",
    "input": {
      "origin": "DEL",
      "destination": "BOM",
      "budget": 6000.0,
      "non_stop": true,
      "preferred_time": null
    },
    "result": [
      "1. search_flights(DEL \u2192 BOM)",
      "2. Filter by budget \u2264 \u20b96,000",
      "3. compare_price() on candidates",
      "4. Apply preferences: non-stop",
      "5. Select best + second-best"
    ],
    "decision"

---
## Edge Case: compare_price with Empty Input

In [10]:
# Test compare_price with empty list
empty_result = compare_price([])
print("compare_price([]) =", json.dumps(empty_result, indent=2))
print()
print("✅ compare_price handles empty input gracefully (no crash)")

compare_price([]) = {
  "cheapest": null,
  "second_best": null,
  "under_budget": [],
  "over_budget": [],
  "nonstop_under_budget": [],
  "price_difference": null,
  "summary": "No options available to compare."
}

✅ compare_price handles empty input gracefully (no crash)
